In [ ]:
"""
fitters_test.ipynb

Tests the fitter object, meant to 
fit all models to a given session.

Author: Stellina X. Ao
Created: 2026-03-05
Last Modified: 2026-03-11
Python Version: 3.11.14
"""

from sg.fitter import LVMFamily
from src.squiggs.neuron_viewer import NeuronViewer
from src.squiggs.renderers import FitRenderer
from sg.eval_models import plot_summary

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt
import pickle
import numpy as np
from pathlib import Path

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
"""
TODO:
backburner
- what about adding ReLU to the taskvar model and removing block?
- plot rasters, sanity check the psths
"""

## Fit one session with 4 additive latents

In [ ]:
trial_data_all = np.load(
    "../vars/trial_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
session_data_all = np.load(
    "../vars/session_data_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
unit_spike_times_all = np.load(
    "../vars/unit_spike_times_all_MM012_MM013_all5.npz", allow_pickle=True
)["arr_0"]
regions_all = np.load("../vars/regions_all_MM012_MM013_all5.npz", allow_pickle=True)[
    "arr_0"
]

subj_idx = 0
sess_idx = 5

unit_spike_times = unit_spike_times_all[subj_idx][sess_idx]
trial_data = trial_data_all[subj_idx][sess_idx]
session_data = session_data_all[subj_idx][sess_idx]
regions = regions_all[subj_idx][sess_idx]
trial_data["block_side"] = np.where(trial_data["block_side"] == "left", 1, -1)

In [ ]:
family = LVMFamily(
    trial_data=trial_data,
    spike_times=unit_spike_times,
    session_data=session_data,
    regions=regions,
    n_latents_mult=1,
    n_latents_addt=1,
    sanity_check=2,
    task_vars=["response", "rewarded", "response_prev", "rewarded_prev"],
    refit=False,
    norm_activity=False,
)
family.fit_all()
family.eval()

In [ ]:
from sg.eval_models import res_taskvar_corr

res_taskvar_corr(family, mode="taskvar")

In [ ]:
plt.figure()  # TODO come back to this
plt.imshow(family.robs.T, aspect="auto")
plt.xlabel("Trials")
plt.ylabel("Neurons")
plt.show()

In [ ]:
renderer = FitRenderer(
    family.mod_offset,
    x=family.test_dl.dataset[:],
    y=family.robs,
    save_subdir=Path("model_fits") / "0312-lm" / "offset",
)

nv = NeuronViewer(num_units=renderer.y.shape[1], render_func=renderer)

In [ ]:
t = family.psths["DMS"].shape[1]
mult = 2 + 2 * np.cumsum(np.random.randn(t)) / (np.arange(t) + 1)

plt.figure()
plt.plot(family.tuber)
plt.show()

In [ ]:
# check if you can get back m2 z-score psths from dividing tuber
# ask joao about the masking of the problem via zscore

"""
- zscore!
    - joao is adamant about this before future analysis
- plot session 1 through 15 instead of clustering by early middle and late
- fit encoder
- or just condition average psths
- check if results match with pca results
    - similar to factor analysis
    - visually check if psths change across the session
- shuffle model-based and model-free labels
- ideally fit individual subspace to separate trial types
- zoom in to session averages, maybe some structure there
- remove below 1 hz
- "why does adding fewer neurons hamper performance, especially with regularization"
- intraregion nonsaturation doesn't hold for striatum
"""

In [ ]:
from scipy.stats import pearsonr as r

potato = (
    2 * np.sin(np.arange(family.psths["DMS"].shape[1]))
    + 0.5 * np.cos(np.arange(family.psths["DMS"].shape[1]))
    + 2.5
)
phase = np.pi / 3
phi = np.sqrt(2)
t = family.psths["DMS"].shape[1]
tuber = (
    np.sin(phi * np.arange(t) / 2 + phase) ** 2
    + 2
    * (0.7 * (np.cos(np.arange(t) / 2 + phase + (np.pi / 2)) + 1) ** 0.75)
    * 0.5
    * np.sin(np.arange(t) / 2) ** 3
    + 2
)

plt.figure()
# plt.plot(potato)
plt.plot(potato)
plt.title(f"$r$={r(potato, tuber).statistic:.3f}")
plt.show()

In [ ]:
yhat = family.mod_gain(family.test_dl.dataset[:])

In [ ]:
from sg.eval_models import plot_cweight_regs, plot_cweights_reg_hist

plt.figure()
plot_cweights_reg_hist(family, family.mod_gain, n_latents=2, mode="gain")
plt.show()

for ax0 in range(3):
    for ax1 in range(3):
        if ax1 > ax0:
            _ = plot_cweight_regs(
                family, family.mod_gain, ax0=ax0, ax1=ax1, num_latents=3, mode="gain"
            )

In [ ]:
plot_summary(family, model=family.mod_affine, potato=family.block_side, mode="affine")

In [ ]:
plt.figure()
X = family.mod_gain.gain_mu.get_weights().T
plt.plot(X[0], color="#2C638A", alpha=0.75, label="Latent 1")
plt.plot(X[1], color="#2BA758", alpha=0.75, label="Latent 2")
plt.plot(X[2], color="#B29016", alpha=0.75, label="Latent 3")
# plt.title(f"$r$={corr[0,1]:.3f}")
plt.xlabel("Trials")
plt.ylabel("Latent Value")
plt.legend()
plt.show()

In [ ]:
model = family.mod_gain
X = model.gain_mu.get_weights().T
X = X - X.mean(axis=1, keepdims=True)
X = X / X.std(axis=1, keepdims=True)

corr = (X @ X.T) / X.shape[1]

plt.figure()
plt.imshow(corr, vmin=-1, vmax=1, cmap="PRGn")
plt.xticks(np.arange(corr.shape[0]), np.arange(corr.shape[0]) + 1)
plt.yticks(np.arange(corr.shape[0]), np.arange(corr.shape[0]) + 1)
plt.xlabel("Latents")
plt.ylabel("Latents")
plt.colorbar(label=r"$r$")
plt.tight_layout()
plt.show()

In [ ]:
render_baseline = FitRenderer(
    family.mod_ae_gain,
    x=family.test_dl.dataset[:],
    y=family.test_dl.dataset[:]["robs"].detach().cpu(),
)
nv = NeuronViewer(num_units=render_baseline.y.shape[1], render_func=render_baseline)

In [ ]:
plot_summary(family, family.mod_offset, potato=family.strategy, mode="offset")

In [ ]:
def get_latent_r(n_m=5, folder="all", do_plot=False):
    import numpy as np

    m_latents = np.arange(n_m + 1)  # 10, 10)
    a_latents = np.arange(5 + 1)  # 10, 10)

    if folder == "all":
        family = LVMFamily(
            trial_data=trial_data,
            spike_times=unit_spike_times,
            session_data=session_data,
            regions=regions,
            n_latents_mult=0,
            n_latents_addt=3,
            sanity_check=0,
            task_vars=[
                "response",
                "rewarded",
                "block_side",
                "response_prev",
                "rewarded_prev",
            ],
            refit=False,
            norm_activity=True,
        )
        family.fit_all()
        family.eval()
    elif folder in regions:
        family = LVMFamily(
            trial_data=trial_data,
            spike_times={folder: unit_spike_times[folder]},
            session_data=session_data,
            regions=[folder],
            n_latents_mult=0,
            n_latents_addt=3,
            sanity_check=0,
            task_vars=[
                "response",
                "rewarded",
                "block_side",
                "response_prev",
                "rewarded_prev",
            ],
            refit=False,
            norm_activity=True,
        )
        family.fit_all()
        family.eval()

    r2s = np.zeros((len(m_latents), len(a_latents)))
    for i, m in enumerate(m_latents):
        for j, a in enumerate(a_latents):
            if m == 0 and a == 0:
                r2s[i, j] = family.res_taskvar["r2test"].mean()
            else:
                with open(
                    f"../vars/families/0312-lm/{folder}/family-m{int(m)}a{int(a)}.pkl",
                    "rb",
                ) as f:
                    family_ = pickle.load(f)
                    family_.eval()
                    r2s[i, j] = family_.res_affine["r2test"].mean()

    if do_plot:
        import matplotlib.pyplot as plt

        plt.style.use(["nature"])
        plt.rcParams["figure.dpi"] = 200
        %matplotlib widget

        plt.figure()
        plt.imshow(r2s, vmin=max(0, np.min(r2s)), origin="lower", interpolation=None)
        plt.xlabel("N. Additive Latents")
        plt.ylabel("N. Multiplicative Latents")
        plt.xticks(np.arange(len(a_latents)))
        plt.yticks(np.arange(len(a_latents)))
        plt.colorbar()
        plt.tight_layout()
        plt.show()

    return r2s


folders = ["all", "ACC", "DMS", "M2", "DLS"]
r2s = {}
for f in folders:
    r2s[f] = get_latent_r(folder=f)

In [ ]:
x = np.where(r2s["all"] == np.max(r2s["all"]))
i = x[0][0]
j = x[1][0]


def get_best_model(folder):
    m, a = np.where(r2s[folder] == np.max(r2s[folder]))
    m = m[0]
    a = a[0]

    if m == 0 and a == 0:
        return
        # NOT IMPLEMENTED FOR NOW
        return None
    else:
        with open(f"../vars/families/0312-lm/{folder}/family-m{m}a{a}.pkl", "rb") as f:
            family = pickle.load(f)
            family.eval()
    return family


families = {}
for folder in folders:
    families[folder] = get_best_model(folder)

In [ ]:
"""
TODO
1. DONE -- plot the r2 maps for all the regions and all 
2. plot the linemaps
3. DONE -- select the best combo for each
4. DONE -- plot the latents
5. DONE -- plot the coupling weights
6. DONE -- plot the correlation map of the latents with each other
7. do some decoding.
"""

In [ ]:
def plot_r2s_line(folder):
    r2 = r2s[folder]

    fig, axes = plt.subplots(nrows=2, ncols=6, figsize=(8, 2))

    for m in range(6):
        r2_slice = r2[m]
        ax = axes[0, m]
        ax.plot(r2_slice)
        ax.set_xticks(np.arange(6))
        ax.set_xlabel("# A. Latents")
        ax.set_ylabel("$r$")
        ax.set_title(f"{m} M. Latents")

    for a in range(6):
        r2_slice = r2[:, a]
        ax = axes[1, a]
        ax.plot(r2_slice)
        ax.set_xticks(np.arange(6))
        ax.set_xlabel("# M. Latents")
        ax.set_ylabel("$r$")
        ax.set_title(f"{a} A. Latents")
    fig.tight_layout()

    from utils.paths import FIGURES_DIR

    save_dir = FIGURES_DIR / "r2s_latents"
    fname = Path("0312-lm") / f"corr-line_{folder}.png"
    fpath = save_dir / fname
    fpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fpath, dpi=300, bbox_inches="tight")


for f in folders:
    plot_r2s_line(f)

In [ ]:
from sg.eval_models import plot_cweight_regs, plot_cweights_reg_hist
from pathlib import Path


def everything(family, folder):
    if family is None:
        return
    if not family.no_mult:
        plot_summary(
            family,
            model=family.mod_affine,
            potato=family.strategy,
            mode="gain",
            save_fig=True,
            fname=Path("0312-lm") / folder / "m-latents.png",
        )
    if not family.no_addt:
        plot_summary(
            family,
            model=family.mod_affine,
            potato=family.strategy,
            mode="offset",
            save_fig=True,
            fname=Path("0312-lm") / folder / "a-latents.png",
        )

    M = family.n_latents_mult
    if not family.no_mult:
        if M == 1:
            plot_cweights_reg_hist(
                family,
                family.mod_affine,
                n_latents=M,
                mode="gain",
                do_save=True,
                fname=Path("0312-lm") / folder / "cweights-m0.png",
            )
        else:
            for ax0 in range(M):
                for ax1 in range(M):
                    if ax1 > ax0:
                        _ = plot_cweight_regs(
                            family,
                            family.mod_affine,
                            ax0=ax0,
                            ax1=ax1,
                            num_latents=M,
                            mode="gain",
                            do_save=True,
                            fname=Path("0312-lm")
                            / folder
                            / f"cweights-m{ax0}m{ax1}.png",
                        )
    A = family.n_latents_addt
    if not family.no_addt:
        if A == 1:
            plot_cweights_reg_hist(
                family,
                family.mod_affine,
                n_latents=A,
                mode="offset",
                do_save=True,
                fname=Path("0312-lm") / folder / "cweights-a0.png",
            )
        else:
            for ax0 in range(A):
                for ax1 in range(A):
                    if ax1 > ax0:
                        _ = plot_cweight_regs(
                            family,
                            family.mod_affine,
                            ax0=ax0,
                            ax1=ax1,
                            num_latents=A,
                            mode="offset",
                            do_save=True,
                            fname=Path("0312-lm")
                            / folder
                            / f"cweights-a{ax0}a{ax1}.png",
                        )

    model = family.mod_affine
    if family.no_mult:
        X = model.offset_mu.get_weights().T
    elif family.no_addt:
        X = model.gain_mu.get_weights().T
    else:
        X = np.concatenate(
            (model.gain_mu.get_weights().T, model.offset_mu.get_weights().T)
        )

    X = X - X.mean(axis=1, keepdims=True)
    X = X / X.std(axis=1, keepdims=True)

    corr = (X @ X.T) / X.shape[1]

    fig, ax = plt.subplots()
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="PRGn")
    ax.set_xticks(np.arange(corr.shape[0]), np.arange(corr.shape[0]) + 1)
    ax.set_yticks(np.arange(corr.shape[0]), np.arange(corr.shape[0]) + 1)
    ax.set_xlabel("Latents")
    ax.set_ylabel("Latents")
    ax.set_title(f"{M} mult latents and {A} addt latents")
    fig.colorbar(im, label=r"$r$")
    fig.tight_layout()
    fig.show()

    from utils.paths import FIGURES_DIR

    save_dir = FIGURES_DIR / "latent_corr"
    fname = Path("0312-lm") / folder / "latent_corr.png"
    fpath = save_dir / fname
    fpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fpath, dpi=300, bbox_inches="tight")

In [ ]:
for folder in folders:
    everything(families[folder], folder)

In [ ]:
families["DMS"] is None

In [ ]:
def plot_strategy(folder):
    family_ = families[folder]

    if family_ is None:
        return

    mx_latents = max(family_.n_latents_mult, family_.n_latents_addt)
    if family_.no_mult or family_.no_addt:
        fig, axes = plt.subplots(
            nrows=1, ncols=mx_latents, figsize=(1.5 * mx_latents, 1.5), squeeze=False
        )
    else:
        fig, axes = plt.subplots(
            nrows=2, ncols=mx_latents, figsize=(1.5 * mx_latents, 2.5)
        )

    if not family_.no_mult:
        for m in range(family_.n_latents_mult):
            ax = axes[0, m]
            mf = family_.mod_affine.gain_mu.get_weights()[(family_.strategy == 0), 0]
            mb = family_.mod_affine.gain_mu.get_weights()[(family_.strategy == 1), 0]
            ax.hist(
                mf, bins=np.linspace(-3, 3, 17), alpha=0.5, color="#222FA9", label="MF"
            )
            ax.hist(
                mb, bins=np.linspace(-3, 3, 17), alpha=0.5, color="#E1A714", label="MB"
            )
            ax.set_title(f"M. Latent {m + 1}")
            ax.legend()
    if not family_.no_addt:
        idx = 1 if not family_.no_mult else 0
        for a in range(family_.n_latents_addt):
            ax = axes[idx, a]
            mf = family_.mod_affine.offset_mu.get_weights()[(family_.strategy == 0), 0]
            mb = family_.mod_affine.offset_mu.get_weights()[(family_.strategy == 1), 0]
            ax.hist(
                mf, bins=np.linspace(-3, 3, 17), alpha=0.5, color="#222FA9", label="MF"
            )
            ax.hist(
                mb, bins=np.linspace(-3, 3, 17), alpha=0.5, color="#E1A714", label="MB"
            )
            ax.set_title(f"A. Latent {a + 1}")
            ax.legend()
    fig.tight_layout()

    from utils.paths import FIGURES_DIR

    save_dir = FIGURES_DIR / "strategy_latent"
    fname = Path("0312-lm") / f"strategy_latent_{folder}.png"
    fpath = save_dir / fname
    fpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fpath, dpi=300, bbox_inches="tight")


for f in folders:
    plot_strategy(f)

## Check latents

In [ ]:
plt.figure()
fig, axes = plt.subplots(ncols=2)
axes[0].imshow(family.train_dl.dataset[:]["dfs"].T)
axes[1].imshow(family.test_dl.dataset[:]["dfs"].T)
plt.show()

In [ ]:
m = 4
a = 10

with open(f"../vars/families/midnight-run/family-m{m}a{a}.pkl", "rb") as f:
    family = pickle.load(f)
family.eval()

plot_summary(family, family.mod_offset, potato=family.strategy, mode="offset")

## Correlate latents

In [ ]:
# m=4
# a=10
# with open(f"../vars/families/midnight-run/family-m{m}a{a}.pkl", "rb") as f:
#     family = pickle.load(f)

# # only focus on offset for now
# model =